# 面试问题：VQ-VAE 怎样用离散码本压缩图像，并避免 codebook collapse？

## 可直接复述的回答主线

1. VQ-VAE 先用 encoder 把图像变成低分辨率连续 latent，再把每个位置替换成距离最近的 codebook 向量。
2. 最近邻 argmin 不可导，因此用 straight-through estimator 把 decoder 梯度原样传给 encoder。
3. codebook loss 只更新离散向量，commitment loss 约束 encoder 输出靠近选中的向量，二者要用 detach 划清梯度路径。
4. 评估不能只看总 loss；要在同一图像上比较简单均值图基线、逐样本重建 MSE、编码网格、码本使用数与 perplexity。
5. 若码本向量初始化完全相同，argmin 会持续偏向第一个 code，出现只有一个 code 被使用的 collapse。
6. 生产还需 EMA 码本更新、dead-code 重启、码率—失真权衡、感知损失、分布漂移和压缩吞吐监控。

后续实验会在同一批输入上依次展示朴素基线、手写核心机制、中间过程、失败修正和生产边界。

## 1. 真实案例与输入预览

案例是 8 张 8×8 二值业务图标：横线、竖线、十字、边框、主对角线、X、T 和棋盘格。它们足够小，可以把原图、重建图和 4×4 离散 code 索引完整打印出来；同时又有明确结构，不是随机张量。

In [1]:
import math  # 汇总梯度范数并解释 perplexity。
import torch  # 使用基础卷积、张量运算和自动微分实现 VQ-VAE。
torch.manual_seed(106)  # 固定网络、码本初始化和训练轨迹。
icon_rows = {"horizontal": ["........", "........", "........", "########", "########", "........", "........", "........"], "vertical": ["...##...", "...##...", "...##...", "...##...", "...##...", "...##...", "...##...", "...##..."], "plus": ["...##...", "...##...", "...##...", "########", "########", "...##...", "...##...", "...##..."], "box": ["########", "#......#", "#......#", "#......#", "#......#", "#......#", "#......#", "########"], "diagonal": ["##......", ".##.....", "..##....", "...##...", "....##..", ".....##.", "......##", ".......#"], "x": ["##....##", ".##..##.", "..####..", "...##...", "...##...", "..####..", ".##..##.", "##....##"], "t": ["########", "########", "...##...", "...##...", "...##...", "...##...", "...##...", "...##..."], "checker": ["#.#.#.#.", ".#.#.#.#", "#.#.#.#.", ".#.#.#.#", "#.#.#.#.", ".#.#.#.#", "#.#.#.#.", ".#.#.#.#"]}  # 定义八张具有明确几何结构的二值图标。
icon_names = list(icon_rows.keys())  # 固定图标展示和训练顺序。
icon_images = torch.tensor([[[[1.0 if character == "#" else 0.0 for character in row] for row in icon_rows[name]]] for name in icon_names], dtype=torch.float32)  # 把字符图标转换为八乘一乘八乘八张量。
def render_icon(image):  # 把浮点图像转为便于阅读的字符画。
    return "\n".join("".join("#" if float(pixel) >= 0.5 else "." for pixel in row) for row in image.squeeze(0))  # 用 0.5 阈值生成八行字符。
print("教学实验输入：8张真实结构图标，tensor shape=", tuple(icon_images.shape))  # 展示图标数量、通道和分辨率。
for name, image in zip(icon_names, icon_images):  # 逐个展示模型实际读取的像素结构。
    print(f"\n{name}\n{render_icon(image)}")  # 输出图标名和完整八乘八字符画。

教学实验输入：8张真实结构图标，tensor shape= (8, 1, 8, 8)

horizontal
........
........
........
########
########
........
........
........

vertical
...##...
...##...
...##...
...##...
...##...
...##...
...##...
...##...

plus
...##...
...##...
...##...
########
########
...##...
...##...
...##...

box
########
#......#
#......#
#......#
#......#
#......#
#......#
########

diagonal
##......
.##.....
..##....
...##...
....##..
.....##.
......##
.......#

x
##....##
.##..##.
..####..
...##...
...##...
..####..
.##..##.
##....##

t
########
########
...##...
...##...
...##...
...##...
...##...
...##...

checker
#.#.#.#.
.#.#.#.#
#.#.#.#.
.#.#.#.#
#.#.#.#.
.#.#.#.#
#.#.#.#.
.#.#.#.#


## 2. Baseline / 基线：所有图只用一张均值图重建

最简单的“压缩”只存训练集逐像素均值，然后无论输入什么都输出同一张图。它能保留常见中心笔画，却无法区分横线、边框和棋盘格。

In [2]:
centroid_image = icon_images.mean(dim=0, keepdim=True)  # 对八张图逐像素取均值得到唯一重建模板。
baseline_reconstructions = centroid_image.repeat(len(icon_names), 1, 1, 1)  # 为每个输入复制完全相同的均值图。
baseline_per_icon_mse = ((baseline_reconstructions - icon_images) ** 2).flatten(1).mean(dim=1)  # 计算每张图相对均值模板的 MSE。
baseline_mse = float(baseline_per_icon_mse.mean().item())  # 汇总同数据基线平均重建误差。
print("Baseline：单一均值图，阈值字符画")  # 标记下方为不区分输入的基线输出。
print(render_icon(centroid_image[0]))  # 展示均值模板阈值化后的结构。
for name, mse in zip(icon_names, baseline_per_icon_mse):  # 逐图展示基线误差而非只给断言。
    print(f"{name:<12} centroid_MSE={mse.item():.5f}")  # 输出当前图标的均值图重建误差。
print(f"Baseline平均MSE={baseline_mse:.5f}")  # 展示后续 VQ-VAE 的同数据参照值。

Baseline：单一均值图，阈值字符画
##.##.#.
.#.#....
...##...
...##..#
#..##...
...##...
....#...
...##..#
horizontal   centroid_MSE=0.18970
vertical     centroid_MSE=0.12720
plus         centroid_MSE=0.17798
box          centroid_MSE=0.29126
diagonal     centroid_MSE=0.17407
x            centroid_MSE=0.22876
t            centroid_MSE=0.16626
checker      centroid_MSE=0.23657
Baseline平均MSE=0.19897


## 3. 底层实现：Encoder、最近邻码本、Straight-Through 与 Decoder

这里不调用现成 VQ-VAE。距离矩阵按 `||z||² + ||e||² - 2ze` 手写；`argmin` 得到 4×4 code 网格；codebook/commitment 两项用不同的 `detach` 控制梯度；decoder 从量化 latent 还原 8×8 图像。

In [3]:
class Encoder(torch.nn.Module):  # 定义把八乘八图像压到四乘四 latent 的卷积编码器。
    def __init__(self, embedding_dim=8):  # 初始化下采样卷积和 latent 投影。
        super().__init__()  # 注册 PyTorch 子模块与参数。
        self.downsample = torch.nn.Conv2d(1, 16, kernel_size=4, stride=2, padding=1)  # 把空间尺寸从八乘八降到四乘四。
        self.project = torch.nn.Conv2d(16, embedding_dim, kernel_size=3, padding=1)  # 把通道映射到八维码本空间。
    def forward(self, images):  # 对一批单通道图像生成连续 latent。
        hidden = torch.relu(self.downsample(images))  # 提取四乘四局部图像特征。
        latents = self.project(hidden)  # 输出批次乘八通道乘四乘四连续 latent。
        return latents  # 返回供最近邻量化的连续表示。
class VectorQuantizer(torch.nn.Module):  # 定义不依赖第三方库的最近邻离散码本。
    def __init__(self, code_count=12, embedding_dim=8, commitment_weight=0.25):  # 初始化码本大小、维度和 commitment 权重。
        super().__init__()  # 注册可训练码本参数。
        self.codebook = torch.nn.Parameter(torch.empty(code_count, embedding_dim))  # 创建十二乘八离散向量表。
        torch.nn.init.uniform_(self.codebook, -0.2, 0.2)  # 用有差异的小随机向量避免初始完全同码。
        self.commitment_weight = commitment_weight  # 保存 encoder 靠近选中 code 的约束强度。
    def forward(self, latents):  # 对每个空间位置执行最近邻查找和直通估计。
        channels_last = latents.permute(0, 2, 3, 1).contiguous()  # 把 latent 排为批次乘高乘宽乘通道。
        flat_latents = channels_last.view(-1, channels_last.shape[-1])  # 展平成所有空间位置乘八维向量。
        distances = (flat_latents ** 2).sum(dim=1, keepdim=True) + (self.codebook ** 2).sum(dim=1).unsqueeze(0) - 2.0 * flat_latents @ self.codebook.T  # 手写每个 latent 到每个 code 的平方欧氏距离。
        flat_indices = distances.argmin(dim=1)  # 为每个位置选择距离最小的离散 code。
        flat_quantized = self.codebook[flat_indices]  # 查表取得选中 code 向量。
        quantized = flat_quantized.view_as(channels_last).permute(0, 3, 1, 2).contiguous()  # 恢复批次乘通道乘高乘宽布局。
        codebook_loss = ((quantized - latents.detach()) ** 2).mean()  # 只让码本向 encoder 输出靠近。
        commitment_loss = ((latents - quantized.detach()) ** 2).mean()  # 只让 encoder 承诺靠近选中码本。
        straight_through = latents + (quantized - latents).detach()  # 前向使用量化值而反向把 decoder 梯度传给 encoder。
        usage = torch.bincount(flat_indices, minlength=self.codebook.shape[0]).to(torch.float32)  # 统计十二个 code 的位置使用次数。
        probabilities = usage / usage.sum().clamp_min(1.0)  # 把使用次数归一化为经验概率。
        nonzero_probabilities = probabilities[probabilities > 0.0]  # 排除零概率避免 log 零。
        perplexity = torch.exp(-(nonzero_probabilities * torch.log(nonzero_probabilities)).sum())  # 计算有效码本复杂度。
        total_vq_loss = codebook_loss + self.commitment_weight * commitment_loss  # 合并标准 VQ 两项损失。
        index_grid = flat_indices.view(latents.shape[0], latents.shape[2], latents.shape[3])  # 恢复每张图四乘四离散索引网格。
        return straight_through, total_vq_loss, {"distances": distances, "indices": index_grid, "usage": usage, "perplexity": perplexity, "codebook_loss": codebook_loss, "commitment_loss": commitment_loss}  # 返回直通 latent、损失和量化证据。
class Decoder(torch.nn.Module):  # 定义把四乘四量化 latent 还原到八乘八图像的解码器。
    def __init__(self, embedding_dim=8):  # 初始化上采样卷积和像素输出层。
        super().__init__()  # 注册解码器参数。
        self.upsample = torch.nn.ConvTranspose2d(embedding_dim, 16, kernel_size=4, stride=2, padding=1)  # 把空间尺寸从四乘四恢复到八乘八。
        self.output = torch.nn.Conv2d(16, 1, kernel_size=3, padding=1)  # 为每个像素输出一个 logit。
    def forward(self, quantized_latents):  # 对量化表示执行图像重建。
        hidden = torch.relu(self.upsample(quantized_latents))  # 上采样并提取局部重建特征。
        reconstruction = torch.sigmoid(self.output(hidden))  # 把像素限制到零到一范围。
        return reconstruction  # 返回批次乘一乘八乘八重建图。
class VQVAE(torch.nn.Module):  # 组合 encoder、离散量化器和 decoder。
    def __init__(self, code_count=12, embedding_dim=8):  # 创建完整离散自编码器。
        super().__init__()  # 注册三个核心模块。
        self.encoder = Encoder(embedding_dim)  # 创建连续图像编码器。
        self.quantizer = VectorQuantizer(code_count, embedding_dim)  # 创建十二项离散码本。
        self.decoder = Decoder(embedding_dim)  # 创建图像解码器。
    def forward(self, images):  # 对一批图像执行编码、量化和重建。
        latents = self.encoder(images)  # 生成连续四乘四 latent。
        straight_through, vq_loss, quantizer_debug = self.quantizer(latents)  # 最近邻量化并保留梯度路径。
        reconstruction = self.decoder(straight_through)  # 从离散表示重建八乘八像素。
        return reconstruction, vq_loss, {"latents": latents, **quantizer_debug}  # 返回重建、VQ 损失和完整中间量。
model = VQVAE()  # 创建待训练的十二 code VQ-VAE。
with torch.no_grad():  # 用真实 encoder latent 做可复现的 data-aware 码本启动。
    initial_latents = model.encoder(icon_images).permute(0, 2, 3, 1).reshape(-1, 8)  # 收集八张图全部一百二十八个位置向量。
    seed_indices = torch.linspace(0, initial_latents.shape[0] - 1, steps=model.quantizer.codebook.shape[0]).long()  # 均匀选择十二个真实 latent 位置。
    model.quantizer.codebook.copy_(initial_latents[seed_indices])  # 用不同业务图位置初始化 codebook，降低冷启动死码风险。
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)  # 创建同时更新 encoder、codebook 和 decoder 的优化器。
history = []  # 保存真实 backward 的损失、梯度和码本使用轨迹。
for step in range(700):  # 在八张结构图标上执行全批次离散重建训练。
    optimizer.zero_grad(set_to_none=True)  # 清除上一步所有参数梯度。
    reconstructions, vq_loss, debug = model(icon_images)  # 执行完整编码、argmin 量化和解码前向。
    reconstruction_loss = ((reconstructions - icon_images) ** 2).mean()  # 计算逐像素重建 MSE。
    loss = reconstruction_loss + vq_loss  # 合并重建、codebook 和 commitment 目标。
    loss.backward()  # 通过 straight-through 对三个模块执行真实反向传播。
    gradient_norm = math.sqrt(sum(float((parameter.grad ** 2).sum().item()) for parameter in model.parameters() if parameter.grad is not None))  # 汇总全部非空梯度的二范数。
    optimizer.step()  # 应用 Adam 更新卷积和码本参数。
    if step % 175 == 0 or step == 699:  # 每一百七十五步保存可读轨迹。
        history.append({"step": step, "loss": loss.item(), "reconstruction": reconstruction_loss.item(), "codebook": debug["codebook_loss"].item(), "commitment": debug["commitment_loss"].item(), "perplexity": debug["perplexity"].item(), "used_codes": int((debug["usage"] > 0).sum().item()), "gradient_norm": gradient_norm})  # 保存损失分项、码本使用和梯度。
with torch.no_grad():  # 取得训练后稳定重建和离散索引。
    vq_reconstructions, final_vq_loss, final_debug = model(icon_images)  # 对八张图执行最终离散重建。
print("VQ-VAE训练轨迹=", history)  # 展示损失下降、非零梯度和码本使用变化。
print("plus连续latent[通道0]=\n", torch.round(final_debug["latents"][2, 0] * 1000) / 1000)  # 展示量化前四乘四连续 latent。
print("plus离散code网格=\n", final_debug["indices"][2])  # 展示同一图像的十六个离散索引。
print("plus首位置到12个code的距离=", torch.round(final_debug["distances"][2 * 16] * 1000) / 1000)  # 展示 argmin 的真实距离依据。

VQ-VAE训练轨迹= [{'step': 0, 'loss': 0.255009263753891, 'reconstruction': 0.2475026696920395, 'codebook': 0.006005274131894112, 'commitment': 0.006005274131894112, 'perplexity': 10.542438507080078, 'used_codes': 12, 'gradient_norm': 0.06451215304998657}, {'step': 175, 'loss': 0.007857105694711208, 'reconstruction': 0.0020640050061047077, 'codebook': 0.004634480457752943, 'commitment': 0.004634480457752943, 'perplexity': 9.687582969665527, 'used_codes': 12, 'gradient_norm': 0.003071028457546279}, {'step': 350, 'loss': 0.0035155699588358402, 'reconstruction': 0.001994486665353179, 'codebook': 0.0012168665416538715, 'commitment': 0.0012168665416538715, 'perplexity': 9.61548900604248, 'used_codes': 12, 'gradient_norm': 0.00062517267769826}, {'step': 525, 'loss': 0.002693332964554429, 'reconstruction': 0.0019756536930799484, 'codebook': 0.0005741434288211167, 'commitment': 0.0005741434288211167, 'perplexity': 9.61548900604248, 'used_codes': 12, 'gradient_norm': 0.00029026715931584646}, {'step':

## 4. 逐图重建结果与结果解读

在完全相同的八张输入上比较单一均值图与 VQ-VAE。除了逐图 MSE，还展示阈值重建字符画、每张图的 4×4 code 网格以及全批次码本占用。

In [4]:
vq_per_icon_mse = ((vq_reconstructions - icon_images) ** 2).flatten(1).mean(dim=1)  # 计算 VQ-VAE 每张图的像素 MSE。
vq_mse = float(vq_per_icon_mse.mean().item())  # 汇总同数据平均重建误差。
healthy_unique_codes = int((final_debug["usage"] > 0).sum().item())  # 统计最终实际使用的离散 code 数。
healthy_perplexity = float(final_debug["perplexity"].item())  # 读取最终码本经验 perplexity。
print("icon         centroid_MSE  VQ_MSE   4x4 codes")  # 输出同数据逐图对照表头。
for index, name in enumerate(icon_names):  # 逐图输出误差与离散表示。
    print(f"{name:<12} {baseline_per_icon_mse[index].item():>12.5f} {vq_per_icon_mse[index].item():>7.5f} {final_debug['indices'][index].tolist()}")  # 展示当前图标基线、模型和 code 网格。
for index in [0, 2, 3, 7]:  # 选择四种差异明显的图标展示可视重建。
    print(f"\n{icon_names[index]} 原图\n{render_icon(icon_images[index])}\n重建\n{render_icon(vq_reconstructions[index])}")  # 输出原图和 VQ 阈值重建字符画。
print(f"结果解读：均值图MSE={baseline_mse:.5f}，VQ-VAE MSE={vq_mse:.5f}；使用{healthy_unique_codes}/12个code，perplexity={healthy_perplexity:.3f}。")  # 解释离散瓶颈在受控图标上的重建与利用率。

icon         centroid_MSE  VQ_MSE   4x4 codes
horizontal        0.18970 0.00000 [[1, 4, 4, 4], [10, 10, 10, 10], [2, 2, 2, 2], [1, 4, 1, 4]]
vertical          0.12720 0.00000 [[8, 3, 10, 8], [4, 3, 10, 8], [4, 3, 10, 8], [4, 3, 7, 4]]
plus              0.17798 0.00001 [[4, 3, 10, 8], [10, 3, 7, 7], [2, 6, 6, 2], [4, 6, 11, 8]]
box               0.29126 0.00002 [[3, 2, 9, 3], [11, 8, 8, 7], [11, 8, 4, 6], [3, 7, 7, 3]]
diagonal          0.17407 0.00001 [[3, 11, 4, 4], [8, 3, 11, 4], [4, 8, 3, 11], [1, 1, 8, 3]]
x                 0.22876 0.01567 [[3, 11, 5, 10], [8, 3, 3, 4], [0, 3, 10, 1], [3, 0, 9, 10]]
t                 0.16626 0.00001 [[3, 3, 3, 3], [1, 2, 7, 8], [4, 3, 10, 8], [4, 3, 7, 4]]
checker           0.23657 0.00002 [[5, 5, 5, 5], [5, 5, 5, 5], [5, 5, 5, 5], [5, 5, 5, 5]]

horizontal 原图
........
........
........
########
########
........
........
........
重建
........
........
........
########
########
........
........
........

plus 原图
...##...
...##...
...##...
########

## 5. 失败案例与修正：把所有 code 初始化成完全相同向量

相同 code 使每个距离并列最小，`argmin` 固定选择索引 0；此时 perplexity=1，其他 code 没有样本、也收不到 codebook 梯度。修正使用来自真实 latent 的分散初始化，并在生产中监控/重启 dead code。

In [5]:
collapsed_quantizer = VectorQuantizer(code_count=12, embedding_dim=8)  # 创建用于复现错误初始化的独立量化器。
with torch.no_grad():  # 禁止失败构造污染训练模型的计算图。
    collapsed_quantizer.codebook.zero_()  # 故意把十二个 code 设成完全相同的零向量。
    trained_latents = model.encoder(icon_images)  # 使用同一批真实图标的训练后连续 latent。
    collapsed_straight, collapsed_loss, collapsed_debug = collapsed_quantizer(trained_latents)  # 在相同 latent 上执行并列距离 argmin。
collapsed_unique_codes = int((collapsed_debug["usage"] > 0).sum().item())  # 统计错误初始化实际使用的 code 数。
collapsed_perplexity = float(collapsed_debug["perplexity"].item())  # 读取错误初始化的码本 perplexity。
dead_code_count = int((final_debug["usage"] == 0).sum().item())  # 统计正确训练后仍未使用的 code，作为生产监控信号。
print("错误行为：相同零向量code使用次数=", collapsed_debug["usage"].to(torch.int64).tolist(), f"unique={collapsed_unique_codes}, perplexity={collapsed_perplexity:.3f}")  # 展示所有位置挤到索引零的 collapse。
print("修正行为：data-aware初始化加联合训练使用次数=", final_debug["usage"].to(torch.int64).tolist(), f"unique={healthy_unique_codes}, perplexity={healthy_perplexity:.3f}")  # 展示多个 code 承担不同局部结构。

错误行为：相同零向量code使用次数= [128, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0] unique=1, perplexity=1.000
修正行为：data-aware初始化加联合训练使用次数= [2, 7, 8, 25, 20, 17, 4, 8, 15, 2, 13, 7] unique=12, perplexity=9.615


## 6. 生产边界

八张 8×8 图只能解释机制。生产图像需要大规模增广数据、分层/残差量化、EMA codebook、dead-code 自动重启、感知或对抗损失、码率—失真曲线、重建公平性评测，以及 encoder FPS、索引带宽、perplexity 和 code 使用漂移监控。

In [6]:
vqvae_diagnostics = {"images": len(icon_names), "resolution": tuple(icon_images.shape[-2:]), "baseline_mse": baseline_mse, "vq_mse": vq_mse, "used_codes": healthy_unique_codes, "dead_codes": dead_code_count, "perplexity": healthy_perplexity, "collapsed_used_codes": collapsed_unique_codes, "collapsed_perplexity": collapsed_perplexity}  # 汇总数据、重建、码本占用和失败指标。
print("生产监控快照：", vqvae_diagnostics)  # 输出离散生成系统应持续观察的信号。

生产监控快照： {'images': 8, 'resolution': (8, 8), 'baseline_mse': 0.198974609375, 'vq_mse': 0.001967440126463771, 'used_codes': 12, 'dead_codes': 0, 'perplexity': 9.61548900604248, 'collapsed_used_codes': 1, 'collapsed_perplexity': 1.0}


## 7. 最小回归测试

最后一格只保留图像规模、真实训练、重建收益、离散索引和 collapse 修正的关键回归断言。

In [7]:
assert len(icon_names) >= 6 and icon_images.shape == (8, 1, 8, 8)  # 保证案例至少包含六张非平凡图像。
assert history[-1]["loss"] < history[0]["loss"] and all(row["gradient_norm"] > 0.0 for row in history)  # 保证 encoder、码本和 decoder 真实 backward 学习。
assert vq_mse < baseline_mse  # 保证同一批图像上离散自编码器优于单一均值图。
assert final_debug["indices"].shape == (8, 4, 4) and int(final_debug["indices"].max().item()) < 12  # 保证每张图得到合法四乘四离散索引。
assert collapsed_unique_codes == 1 and collapsed_perplexity == 1.0  # 保证相同码本初始化的 collapse 可复现。
assert healthy_unique_codes >= 3 and healthy_perplexity > collapsed_perplexity  # 保证分散初始化和训练恢复多个有效 code。